# YOLOv11 Local Training and Testing Notebook
## Real-time Student Behavior Detection (Local)

This notebook is local-first (Windows/venv friendly). Run cells top to bottom.

## 1. Install and Import Dependencies

In [1]:
import sys
import platform
import subprocess

print('Python:', sys.version)
print('Executable:', sys.executable)
print('Platform:', platform.platform())

def ensure_package(pkg: str) -> None:
    try:
        __import__(pkg)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

ensure_package('ultralytics')
ensure_package('yaml')
ensure_package('cv2')
ensure_package('matplotlib')
ensure_package('PIL')

Python: 3.12.6 (tags/v3.12.6:a4a2d2b, Sep  6 2024, 20:11:23) [MSC v.1940 64 bit (AMD64)]
Executable: d:\FYP\FYP CODE\.venv\Scripts\python.exe
Platform: Windows-11-10.0.26200-SP0


In [2]:
from pathlib import Path
import json
import yaml
import torch
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

## 2. Configure Local Paths

In [3]:
PROJECT_ROOT = Path('.').resolve()
DATASET_ROOT = PROJECT_ROOT / 'dataset'
DATA_YAML = DATASET_ROOT / 'data.yaml'

# If you copied fyp_runs from Colab, keep it in project root.
RUNS_ROOT = PROJECT_ROOT / 'fyp_runs'
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR = RUNS_ROOT / 'classroom_model_v1'
BEST_WEIGHTS = RUN_DIR / 'weights' / 'best.pt'
LAST_WEIGHTS = RUN_DIR / 'weights' / 'last.pt'

device = '0' if torch.cuda.is_available() else 'cpu'

print(f'Project root: {PROJECT_ROOT}')
print(f'Dataset root: {DATASET_ROOT}')
print(f'Data yaml: {DATA_YAML}')
print(f'Runs root: {RUNS_ROOT}')
print(f'PyTorch version: {torch.__version__}')
print(f'torch.cuda.is_available(): {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device name: {torch.cuda.get_device_name(0)}')
else:
    print('CUDA GPU not detected. Training will run on CPU unless configured otherwise.')

Project root: D:\FYP\FYP CODE
Dataset root: D:\FYP\FYP CODE\dataset
Data yaml: D:\FYP\FYP CODE\dataset\data.yaml
Runs root: D:\FYP\FYP CODE\fyp_runs
PyTorch version: 2.11.0+cpu
torch.cuda.is_available(): False
CUDA GPU not detected. Training will run on CPU unless configured otherwise.


In [ ]:
if not DATA_YAML.exists():
    raise FileNotFoundError(f'Missing file: {DATA_YAML}')

expected_dirs = [
    DATASET_ROOT / 'train' / 'images',
    DATASET_ROOT / 'train' / 'labels',
    DATASET_ROOT / 'valid' / 'images',
    DATASET_ROOT / 'valid' / 'labels',
    DATASET_ROOT / 'test' / 'images',
    DATASET_ROOT / 'test' / 'labels',
]

print('Dataset directory check:')
all_ok = True
for p in expected_dirs:
    ok = p.exists()
    print(f"{'OK ' if ok else 'MISS'} - {p}")
    all_ok = all_ok and ok

if not all_ok:
    raise FileNotFoundError('One or more dataset directories are missing.')

## 3. Validate Dataset YAML

In [4]:
with open(DATA_YAML, 'r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)

normalize_map = {
    '../train/images': 'train/images',
    '../valid/images': 'valid/images',
    '../test/images': 'test/images',
}
changed = False
for k in ['train', 'val', 'test']:
    cur = data_cfg.get(k)
    if cur in normalize_map:
        data_cfg[k] = normalize_map[cur]
        changed = True

if changed:
    with open(DATA_YAML, 'w', encoding='utf-8') as f:
        yaml.safe_dump(data_cfg, f, sort_keys=False)
    print('Updated data.yaml paths to local-relative format.')

print('Loaded data.yaml:')
print(json.dumps({
    'train': data_cfg.get('train'),
    'val': data_cfg.get('val'),
    'test': data_cfg.get('test'),
    'nc': data_cfg.get('nc'),
    'names': data_cfg.get('names'),
}, indent=2))

Loaded data.yaml:
{
  "train": "train/images",
  "val": "valid/images",
  "test": "test/images",
  "nc": 6,
  "names": [
    "handrise",
    "read",
    "sleep",
    "stand",
    "using_electronic_devices",
    "write"
  ]
}


## 4. Train (Optional Local Run)

In [ ]:
# Set DO_TRAIN=True if you really want local training.
DO_TRAIN = False
RESUME_IF_AVAILABLE = False

if DO_TRAIN:
    if device == 'cpu':
        print('Running on CPU. Consider smaller settings for faster turnaround.')

    model = YOLO(str(LAST_WEIGHTS if RESUME_IF_AVAILABLE and LAST_WEIGHTS.exists() else Path('yolo11n.pt')))

    QUICK_RUN = True
    if QUICK_RUN:
        epochs = 5
        imgsz = 512
        batch = 8
        patience = 5
    else:
        epochs = 30
        imgsz = 640
        batch = 16
        patience = 10

    train_config = {
        'data': str(DATA_YAML),
        'epochs': epochs,
        'imgsz': imgsz,
        'batch': batch,
        'device': device,
        'project': str(RUNS_ROOT),
        'name': 'classroom_model_v1',
        'verbose': True,
        'save': True,
        'workers': 2,
        'cache': False,
        'amp': True,
        'optimizer': 'auto',
        'cos_lr': True,
        'patience': patience,
        'plots': True,
        'close_mosaic': 10,
        'seed': 42,
    }

    print('Training config:')
    for k, v in train_config.items():
        print(f'  {k}: {v}')

    model.train(**train_config)
    print('Training complete.')
else:
    print('Training skipped. Set DO_TRAIN=True to train locally.')

## 5. Test Saved Best Weights

In [7]:
if not BEST_WEIGHTS.exists():
    raise FileNotFoundError(f'Missing trained weights: {BEST_WEIGHTS}. Copy your fyp_runs folder first or train locally.')

test_model = YOLO(str(BEST_WEIGHTS))

test_config = {
    'data': str(DATA_YAML),
    'split': 'test',
    'imgsz': 640,
    'batch': 16,
    'device': device,
    'project': str(RUNS_ROOT),
    'name': 'classroom_model_v1_test',
    'verbose': True,
    'save': True,
    'plots': True,
}

print('Testing config:')
for k, v in test_config.items():
    print(f'  {k}: {v}')

test_results = test_model.val(**test_config)
print('Testing complete.')

results_dir = Path(getattr(test_results, 'save_dir', RUNS_ROOT / 'classroom_model_v1_test'))
print(f'Test artifacts saved to: {results_dir}')

if hasattr(test_results, 'results_dict'):
    print('Metrics:')
    print(json.dumps(test_results.results_dict, indent=2, default=str))

Testing config:
  data: D:\FYP\FYP CODE\dataset\data.yaml
  split: test
  imgsz: 640
  batch: 16
  device: cpu
  project: D:\FYP\FYP CODE\fyp_runs
  name: classroom_model_v1_test
  verbose: True
  save: True
  plots: True
Ultralytics 8.4.30  Python-3.12.6 torch-2.11.0+cpu CPU (AMD Ryzen 5 2600 Six-Core Processor)
YOLO11n summary (fused): 101 layers, 2,583,322 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 5.85.4 ms, read: 2.80.6 MB/s, size: 50.0 KB)
val: Scanning D:\FYP\FYP CODE\dataset\test\labels... 771 images, 9 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 771/771 291.0it/s 2.6s0.1s
val: New cache created: D:\FYP\FYP CODE\dataset\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 49/49 1.3s/it 1:061.3sss
                   all        771       2617      0.808      0.824      0.839      0.513
              handrise        413        595      0.912      0.908      0.927      0.604
             

## 6. Visualize Results

In [8]:
results_png = RUN_DIR / 'results.png'
cm_png = RUN_DIR / 'confusion_matrix.png'
val_batch_png = RUN_DIR / 'val_batch0_pred.jpg'

print('Training artifacts:')
for p in [results_png, cm_png, val_batch_png]:
    print(f"{'OK ' if p.exists() else 'MISS'} - {p}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
images = [results_png, cm_png, val_batch_png]
titles = ['Training Curves', 'Confusion Matrix', 'Sample Validation Predictions']

for ax, img_path, title in zip(axes, images, titles):
    ax.set_title(title)
    ax.axis('off')
    if img_path.exists():
        ax.imshow(Image.open(img_path))
    else:
        ax.text(0.5, 0.5, 'Not generated yet', ha='center', va='center')

plt.tight_layout()
plt.show()

Training artifacts:
OK  - D:\FYP\FYP CODE\fyp_runs\classroom_model_v1\results.png
OK  - D:\FYP\FYP CODE\fyp_runs\classroom_model_v1\confusion_matrix.png
OK  - D:\FYP\FYP CODE\fyp_runs\classroom_model_v1\val_batch0_pred.jpg


<Figure size 1800x500 with 3 Axes>